<a href="https://colab.research.google.com/github/rohanraaj2/Human-Computer-Interaction-and-Explainable-AI/blob/main/Shap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SHAP Exercise:
#### **How Do Different SHAP Methods Explain the Same Image?**

SHAP has different **explainers** that use different strategies to compute the Shapley values. In this exercise, you'll apply different explainers to the same image and compare their explanations.

## Setup and Imports
Start by setting up a new virtual environment with the packages specified in requirements_shap.txt.
I tested the scripts with python 3.11 but newer versions probably also work.

Then download the *poodle.png* and *imagenet_classes.txt* from Moodle into a folder called Data. Since we will reuse this folder, you can create it one level above the folder for this SHAP exercise.

Then:

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import requests
from skimage.segmentation import slic
import shap
import cv2

device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
# Load pre-trained VGG16
model = models.vgg16(weights="IMAGENET1K_V1")
model.eval()
model = model.to(device)

# Load ImageNet class names
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
response = requests.get(url)
class_names = [line.strip() for line in response.text.split('\n')]
print(f"Model loaded. {len(class_names)} classes.")

In [ ]:
# Path to images
data_path = Path("../Data")
img_name = "poodle.png"

# Load image
img_path = data_path / img_name
if img_path.exists():
    original_img = Image.open(img_path).convert('RGB')
else:
    print("Image not found, creating placeholder")
    original_img = Image.new('RGB', (224, 224), color='gray')

original_img = original_img.resize((224, 224))
img_array = np.array(original_img)

plt.imshow(original_img)
plt.title(f"Test Image: {img_name}")
plt.axis('off')
plt.show()

In [ ]:
# Preprocessing for Imagenet like data
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Get model prediction
img_tensor = preprocess(original_img).unsqueeze(0).to(device)
with torch.no_grad():
    output = model(img_tensor)
    probs = torch.nn.functional.softmax(output[0], dim=0)
    top5 = torch.argsort(probs, descending=True)[:5]

predicted_class = top5[0].item()
print(f"\nPredicted: {class_names[predicted_class]} ({probs[predicted_class]:.3f})")
print("\nTop 5 predictions:")
for i, idx in enumerate(top5):
    print(f"  {i+1}. {class_names[idx]}: {probs[idx]:.3f}")

## Prediction Function
The SHAP library implements different variations of SHAP in functions called *explainers*.
All SHAP explainers need a function that takes images and returns predictions.

In [ ]:
def model_predict(images):
    """
    Predict class probabilities for a batch of images.

    Args:
        images: numpy array of shape (n_samples, H, W, 3) with values in [0, 255]

    Returns:
        numpy array of shape (n_samples, n_classes) with probabilities
    """
    # TODO: Implement this function
    images_processed = [] # We add all the numpy images we have preprocessed here

    for img in images: # Going through our numpy image batch
        img = img.astype(np.uint8) # Here we are making sure that the datatype is uint8 for the python imaging library(PIL) to ensure PIL doesnt give error or read incorrectly

        # Now we convert the numpy array or well image to PIL image
        pil_img = Image.fromarray(img) # We need an image and img is just an array of numbers, so we use x.fromarray(y) to convert the array(img) into an actual PIL image

        tensor_img = preprocess(pil_img) # Here we are applying same preprocessing pipeline as we did before
        # Now our image is model ready. Basically it's not an image now but a pytorch tensor

        images_processed.append(tensor_img) # Adding the ready image

    batch = torch.stack(images_processed).to(device) # Combines all images into one batch and sends to the CPU

    # Model prediction
    with torch.no_grad(): # We dont want pytorch to calculate gradients because we are only doing predictions not actual training right now
        output = model(batch) # Returns raw scores from the neural network

        probs = torch.nn.functional.softmax(output, dim=1) # Convert those raw scores into actual probabilities

    return probs.cpu().numpy() # Converts the tensor back to numpy array

# Test the function
test_input = np.array([img_array])
test_output = model_predict(test_input)
print(f"Test output shape: {test_output.shape}")
print(f"Probability for {class_names[predicted_class]}: {test_output[0, predicted_class]:.3f}")

# KernelSHAP
We will start with the basic KernelSHAP explainer which we saw in the lecture.

**Reminder how it works**: Kernel SHAP approximates Shapley values by sampling coalitions of features (here, superpixels) and weighting them with a Shapley kernel; it is model‑agnostic but relatively slow.

## Masking

The SHAP library needs a masking function to represent "feature absent" perturbation.
For our image data and KernelSHAP, we will use a superpixel segmentation approach similar to LIME, as discussed in the lecture.
To this end, we first segement the image using the SLIC algorithm. If you want, you can play around with the hyperparameters of SLIC.

In [ ]:
# Superpixels
n_segments = 50 # Number of superpixels
segments = slic(img_array, n_segments=n_segments, compactness=30, sigma=3, start_label=0) # This returns a 2D integer array assigning a superpixel id to each pixel.


In addition, SHAP requires a background data for perturbation. In our case this background data simply defines the "color" that is used to perturb the superpixels.
Let's start with a simple perturbation using only black color:

In [ ]:
background = np.zeros((1, 224, 224, 3), dtype=np.uint8)

Now, we can create custom masking functions to perturb the image and get the models prediction on this perturbed sample.

In [ ]:
def mask_image_kernel(masks, segmentation, image, background):
    '''
    A utility masking function for Kernel SHAP that converts binary coalition masks
    over superpixels into perturbed images.

    :param masks: A 2D binary array of shape (n_masks, n_segments) where the entries of each mask
        indicate which superpixels are kept for the masked sample.
    :param segmentation: A 2D integer array assigning each pixel to a superpixel id.
        Superpixel ids are expected to match the column indices of `masks`.
    :param image: The original input image of shape (H, W, 3).
    :param background: The baseline image used to fill masked-out superpixels.
        Must have the same shape as `image`.
    :return: A batch of perturbed images of shape (n_masks, H, W, 3), where each
        image corresponds to one coalition mask.
    '''
    # TODO YOUR CODE HERE

    # Here we are instead manipulating/masking pixels indivdiually, we are doing neighbouring clusters instead. Those clusters are super-pixels

    # This function will now take binary masks and turn them into actual masked images

    images_pertubed = [] # We will store all generated mask images here

    b_g = background[0] # Background has a batch dimension (1, H, W, 3) so we are indexing with [0] to remove the batch dimension and gives us the actual RGB image

    for item in masks: # Here we are looping over every coalition mask

        perturbed_image = image.copy() # We are starting with a copy of the original image and then applying mask by filling the missing superpixels with b_g

        for super_pixel, keep in enumerate(item):

            if keep == 0: # Flag to check it we should keep super_pixel or not

                perturbed_image[segmentation == super_pixel] = b_g[segmentation == super_pixel] # Since we don't need to keep that super_pixel, we can replace all its pixels with the black bakgorund

        images_pertubed.append(perturbed_image) # Saving the final image that we process

    return np.array(images_pertubed) # Returning the array

def f_kernel(masks):
    '''
    A utility prediction function that uses binary masks to predict different coalitions.
    :param masks: A binary mask [0,1,0,...] that represents which superpixels are present.
    :return: The classification for the perturbed image.
    '''
    # TODO YOUR CODE HERE
    images_pertubed_2 = mask_image_kernel(masks, segments, img_array, background) # Applying our prev function

    return model_predict(images_pertubed_2) # Returning the models's prediction on our perturbed images

## KernelExplainer

Now that we have the necessary auxiliary functions, let's create our SHAP explanation.
Use the *KernelExplainer* class and its *shap_values* method to explain our poodle image.
Hint: The KernelExplainer expects the background argument in binary coalition mask form.
- See SHAP docs: [shap.KernelExplainer](https://shap.readthedocs.io/en/latest/generated/shap.KernelExplainer.html)


In [ ]:
# TODO: Create and run KernelExplainer
# YOUR CODE HERE

'''
We are using a helper function:
- SHAP for KernelExplainer will give us values per superpixel
- But if we want to plot , then we need a pixel-level heat map
- So this function spreads each superpixel SHAP value back to all pixels in that region so we can plot
'''

def helper_funct(shap_values, segmentation):

    # Empty 2D array same size as segmentation map
    pixel_map = np.zeros(segmentation.shape, dtype=np.float32)

    # We are now going loop through each super pixel ID and shap value
    for sup_pix_id, shap_val in enumerate(shap_values):
        pixel_map[segmentation == sup_pix_id] = shap_val # Wherever segmentation map says that pixel belongs to this superpixel, we assign the shap value to those pixels
    return pixel_map

# We know that KernelExplainer expects a binary coalition background mask, not the actual image background. Thus, here we make one base sample where all superpixels are absent = basically all zeros

bg_masks = np.zeros((1, n_segments), dtype=np.float32)

# For kernel explainer object
explainer_obj = shap.KernelExplainer(f_kernel, bg_masks) # Here we are creating a kernel explainer object

# Now we are explaining the full image
# All ones means every superpixel is present
test_mask = np.ones((1, n_segments), dtype=np.float32)

# TODO: Extract shap values for predicted class

# SHAP will now sample different coalitions internally and estimates the shap vals
# n samples can be increased for better result but it becomes slower

shap_values_2 = explainer_obj.shap_values(test_mask, nsamples=100)

# For multiclass classification shap will usually only return a list with one entry per class
# Now we only want the shap explanation for the class the model actually predicted

shap_values_final = shap_values_2[0, :, predicted_class] # Fix for out of index error

Now, use the shap.image_plot method to visualize your result.
Hint: you might need another helper function.
- See SHAP docs: [shap.image_plot]( https://shap.readthedocs.io/en/latest/generated/shap.plots.image.html)

In [ ]:
# Visualize KernelExplainer SHAP values
# TODO YOUR CODE HERE

pixel_shap = helper_funct(shap_values_final, segments) # Converting superpixel shap values to pixel shap map

pixel_shap_batch = np.expand_dims(pixel_shap, axis=(0, -1)) # Adding batch and channel dimensions because shap.image_plot likes image-shaped batch input
image_batch = np.expand_dims(img_array, axis=0)

shap.image_plot([pixel_shap_batch], image_batch) # Visualizing shap explanation

## Different Background
Now lets try a random background for perturbation instead of black.
Create a background image with random pixels (an image where each pixel has random values between 0 and 255) and use this for our custom masking function.
Then, see how this changes the explanation for your example image.

In [ ]:
# TODO YOUR CODE HERE
'''
- Now instead of black background, create a random-noise background image
- Each pixel gets random RGB values between 0 and 255
'''

background = np.random.randint(0, 256, size=(1, 224, 224, 3), dtype=np.uint8)

bg_masks_2 = np.zeros((1, n_segments), dtype=np.float32) # Again KernelExplainer still wants binary coalition background format

explainer_random_bg = shap.KernelExplainer(f_kernel, bg_masks_2) # Recreating explainer so it now uses the new random image background inside f_kernel with masking

random_shap_values = explainer_random_bg.shap_values(test_mask, nsamples=100) # Calculating shap values again for the same full image

predicted_rand_shap_vals = random_shap_values[0, :, predicted_class] # Again only keeping shap values for the predicted class

pixel_shap_random = helper_funct(predicted_rand_shap_vals, segments) # Converting superpixel values back to pixel map to plot

pixel_shap_random_batch = np.expand_dims(pixel_shap_random, axis=(0, -1)) # Again adding dimensions for plotting

shap.image_plot([pixel_shap_random_batch], image_batch) # Visualizing explanation with random background

# PartitionSHAP

Now, we will look at a different way to calculate SHAP values: the PartitionExplainer.

**How it works**: Uses a hierarchical feature partition tree and a masking function (for images, shap.maskers.Image) to efficiently approximate Shapley values, making it much faster than kernel-based methods for structured inputs.
See SHAP docs:
- [shap.PartitionExplainer](https://shap.readthedocs.io/en/latest/generated/shap.PartitionExplainer.html)
- [shap.maskers.Image](https://shap.readthedocs.io/en/latest/generated/shap.maskers.Image.html)

## ParitionExplainer with Inpainting

In the first PartitionExplainer example, we will use inpainting to mask the images. Here the masking tries to reconstruct the selected image area from the pixels near the area boundary. This leads to more realistic perturbations than using a constant color.

In [ ]:
# Image masker
masker = shap.maskers.Image("inpaint_telea", img_array.shape)

# TODO: Create Partition explainer
# YOUR CODE HERE

# TODO: Compute SHAP values
# YOUR CODE HERE

In [ ]:
# TODO: Visualize PartitionExplainer SHAP values using shap.image_plot
# YOUR CODE HERE

## PartitionExplainer with Blur Masking

Now, let's try to use an image masker with a blur value (e.g. "blur(128,128)") so that masked regions are replaced by a blurred version of the image, representing “feature absent” via smoothing instead of inpainting or a fixed background.
- See SHAP docs: [shap.maskers.Image](https://shap.readthedocs.io/en/latest/generated/shap.maskers.Image.html)


In [ ]:
# TODO: Adjust the image masker from the previous example to use 128x128 blurring
# YOUR CODE HERE

# TODO: Create Partition explainer
# YOUR CODE HERE

# TODO: Compute SHAP values
# YOUR CODE HERE

In [ ]:
# TODO: Visualize PartitionExplainer SHAP values using shap.image_plot
# YOUR CODE HERE